# TimelyMT P3-GLOBAL Checkpoint Manager

> **TRAIN and DEV only. TEST is forbidden.** Run All validates/restores only; it never trains, rolls out, evaluates, or publishes.

The frozen upstream Dataset supplies only V1 TRAIN pseudo-label supervision. P3 state is restored and published only through its dedicated Dataset.

## 0. Operator Configuration

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

# Edit this cell only when changing a Kaggle experiment.
REPOSITORY_URL = 'https://github.com/MinhCYB/TimelyMT.git'
REPOSITORY_REF = 'main'
UPSTREAM_CHECKPOINT_DATASET_REF = 'iteams24/timelymt-research-checkpoints'
P3_CHECKPOINT_DATASET_REF = 'iteams24/timelymt-p3-global-checkpoints'
FORCE_RETRAIN_P3 = False
RESTORE_P3_CHECKPOINT = True
PUBLISH_P3_CHECKPOINT = False

WORKING_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO_ROOT = WORKING_ROOT / 'TimelyMT'
SRC_ROOT = REPO_ROOT / 'src'
CONFIG_PATH = REPO_ROOT / 'configs/experiments/policy-p3-global.json'
PREPARED_MANIFEST = REPO_ROOT / 'data/prepared_context/manifest.json'
PSEUDO_TRAIN = REPO_ROOT / 'data/policy/pseudo_labels/train'
DOWNLOAD_ROOT = WORKING_ROOT / 'p3-checkpoint-downloads'
P3_PACKAGE_ARCHIVE = WORKING_ROOT / 'timelymt-p3-global-checkpoint.tar.gz'
assert not PUBLISH_P3_CHECKPOINT, 'Publishing must be an explicit manual action.'

## 1. Repository and Package Setup

In [ ]:
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPOSITORY_REF, '--single-branch', REPOSITORY_URL, str(REPO_ROOT)], check=True)
if str(SRC_ROOT) not in sys.path: sys.path.insert(0, str(SRC_ROOT))
os.environ['PYTHONPATH'] = str(SRC_ROOT)
if not CONFIG_PATH.is_file() or not PREPARED_MANIFEST.is_file(): raise FileNotFoundError('P3 source/config/prepared-context contract is incomplete')
from timelymt.research.p3_checkpointing import (
    build_p3_package, discover_p3_candidates, local_p3_checkpoint,
    repository_identity, resolve_local_conflict, restore_p3_candidate,
    restore_upstream_supervision, validate_p3_candidate,
)
from timelymt.research.policy_p3_global import prepared_manifest_fingerprint
from timelymt.research.policy_v2 import validate_v1_supervision
print({'commit': repository_identity(REPO_ROOT)['repo_commit'], 'prepared_manifest': prepared_manifest_fingerprint(PREPARED_MANIFEST)})

## 2. Safe Dataset Download Helpers

Dataset downloads are input operations only. The upstream Dataset is never published or altered.

In [ ]:
def kaggle_download(dataset_ref, destination):
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_ref, '-p', str(destination), '--unzip'], text=True, capture_output=True)
    if result.returncode:
        print(f'Dataset unreachable or empty: {dataset_ref}: {result.stderr.strip() or result.stdout.strip()}')
        return False
    return True

def restore_upstream_if_needed():
    try:
        manifest, rows = validate_v1_supervision(PSEUDO_TRAIN, 'train')
        return {'restored': False, 'manifest': manifest, 'rows': rows}
    except RuntimeError:
        pass
    upstream = DOWNLOAD_ROOT / 'upstream'; shutil.rmtree(upstream, ignore_errors=True)
    if not kaggle_download(UPSTREAM_CHECKPOINT_DATASET_REF, upstream):
        raise RuntimeError('Required frozen V1 TRAIN supervision is absent and could not be restored.')
    candidates = sorted(upstream.rglob('*.tar.gz')) + sorted({p.parent for p in upstream.rglob('manifest.json')})
    for candidate in candidates:
        try:
            restore_upstream_supervision(candidate, REPO_ROOT)
            manifest, rows = validate_v1_supervision(PSEUDO_TRAIN, 'train')
            return {'restored': True, 'manifest': manifest, 'rows': rows}
        except RuntimeError:
            continue
    raise RuntimeError('Frozen upstream Dataset had no safe, valid TRAIN pseudo-label package.')

UPSTREAM_STATUS = restore_upstream_if_needed()

## 3. Automatic P3 Restore

Candidates may be raw `.tar.gz` files or Kaggle-expanded directories. Each is fully validated in a temporary location before any repository file is replaced.

In [ ]:
P3_RESTORE_STATUS = {'found': False, 'compatible': False, 'stage': 'NONE', 'created_at': None, 'action': 'not-requested'}
if RESTORE_P3_CHECKPOINT:
    persistent_root = DOWNLOAD_ROOT / 'p3'; shutil.rmtree(persistent_root, ignore_errors=True)
    reachable = kaggle_download(P3_CHECKPOINT_DATASET_REF, persistent_root)
    candidates = discover_p3_candidates(persistent_root, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST) if reachable else []
    if not candidates:
        print('No compatible P3 checkpoint found.')
        P3_RESTORE_STATUS.update({'reachable': reachable, 'action': 'none'})
    else:
        selected = candidates[0]; local = local_p3_checkpoint(REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
        action = resolve_local_conflict(local, selected['metadata'])
        if action == 'conflict': raise RuntimeError('Local and persistent valid P3 checkpoints differ with ambiguous ordering; ask the researcher before replacing either.')
        if action == 'restore-persistent': restore_p3_candidate(selected['path'], REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
        P3_RESTORE_STATUS.update({'found': True, 'compatible': True, 'reachable': reachable, 'action': action, **selected['metadata']})
        print('Restored P3 checkpoint:' if action == 'restore-persistent' else 'Keeping valid local P3 checkpoint:')
        print({key: P3_RESTORE_STATUS.get(key) for key in ('completed_stage', 'checkpoint_sha256', 'created_at', 'repo_commit', 'prepared_context_manifest_fingerprint')})
P3_LOCAL = local_p3_checkpoint(REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
P3_CHECKPOINT_VALID = P3_LOCAL is not None

## 4. P3 Session Status

In [ ]:
prepared = json.loads(PREPARED_MANIFEST.read_text(encoding='utf-8'))
identity = repository_identity(REPO_ROOT)
print('P3 SESSION STATUS')
print(f"Repository:\n  commit: {identity['repo_commit']}\n  dirty: {identity['working_tree_dirty']}")
print(f"Upstream supervision:\n  restored: {'YES' if UPSTREAM_STATUS['restored'] else 'YES (already local)'}\n  manifest: {PSEUDO_TRAIN / 'manifest.json'}\n  TRAIN rows: {len(UPSTREAM_STATUS['rows'])}")
print(f"Prepared context:\n  manifest fingerprint: {prepared_manifest_fingerprint(PREPARED_MANIFEST)}\n  TRAIN context talks: {sum(1 for x in prepared['pools'] if x['split'] == 'train' and x.get('sources'))}\n  DEV context talks: {sum(1 for x in prepared['pools'] if x['split'] == 'dev' and x.get('sources'))}")
print(f"P3 persistent dataset:\n  ref: {P3_CHECKPOINT_DATASET_REF}\n  reachable: {'YES' if P3_RESTORE_STATUS.get('reachable') else 'NO'}")
print(f"P3 checkpoint:\n  found: {'YES' if P3_RESTORE_STATUS['found'] else 'NO'}\n  compatible: {'YES' if P3_CHECKPOINT_VALID else 'NO'}\n  stage: {P3_RESTORE_STATUS.get('completed_stage', 'NONE')}\n  created_at: {P3_RESTORE_STATUS.get('created_at')}")
print(f"TRAIN required: {'NO' if P3_CHECKPOINT_VALID and not FORCE_RETRAIN_P3 else 'YES'}")

## 5. Manual TRAIN P3

This cell is safe under Run All. Set `RUN_TRAIN_P3 = True` and, if replacing a valid checkpoint intentionally, `FORCE_RETRAIN_P3 = True` in the configuration cell.

In [ ]:
RUN_TRAIN_P3 = False
if RUN_TRAIN_P3:
    if P3_CHECKPOINT_VALID and not FORCE_RETRAIN_P3:
        raise RuntimeError('A compatible P3 checkpoint already exists. Set FORCE_RETRAIN_P3=True intentionally to train from scratch.')
    if repository_identity(REPO_ROOT)['working_tree_dirty']:
        print('WARNING: checkout contains uncommitted changes; confirm scientific-code identity before training.')
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'train-p3'], cwd=REPO_ROOT, check=True)
else:
    print('TRAIN is disabled. Set RUN_TRAIN_P3=True intentionally after reviewing status.')

## 6. Post-Training Checkpoint Inspection

The official internal validator and CLI inspection must pass before persistence.

In [ ]:
P3_LOCAL = local_p3_checkpoint(REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
if P3_LOCAL:
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'inspect-p3-checkpoint'], cwd=REPO_ROOT, check=True)
    internal = P3_LOCAL['internal']
    print({'variant': internal['variant'], 'input_dimension': internal['input_dimension'], 'training_completion': 'validated local checkpoint', 'manifest_fingerprint': internal['prepared_context_manifest_fingerprint'], 'checkpoint_hash': internal['checkpoint_sha256']})
else:
    print('No compatible P3 checkpoint restored.')

## 7. Manual P3 Dataset Publication

Call `publish_p3_checkpoint(stage='TRAINED')` manually after training and inspection. It publishes a new version only to `P3_CHECKPOINT_DATASET_REF`, never to the frozen upstream Dataset.

In [ ]:
def publish_p3_checkpoint(*, stage='TRAINED'):
    if not PUBLISH_P3_CHECKPOINT:
        raise RuntimeError('Publishing is disabled. Set PUBLISH_P3_CHECKPOINT=True and call this function intentionally.')
    build_p3_package(REPO_ROOT, P3_PACKAGE_ARCHIVE, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST, stage=stage)
    upload = WORKING_ROOT / 'p3-checkpoint-upload'; shutil.rmtree(upload, ignore_errors=True); upload.mkdir()
    shutil.copy2(P3_PACKAGE_ARCHIVE, upload / P3_PACKAGE_ARCHIVE.name)
    (upload / 'dataset-metadata.json').write_text(json.dumps({'title': 'TimelyMT P3 GLOBAL Checkpoints', 'id': P3_CHECKPOINT_DATASET_REF, 'licenses': [{'name': 'other'}]}, indent=2))
    subprocess.run(['kaggle', 'datasets', 'version', '-p', str(upload), '-m', f'P3_GLOBAL {stage}', '--dir-mode', 'zip'], check=True)
    print(f'Published P3 checkpoint to {P3_CHECKPOINT_DATASET_REF}: {P3_PACKAGE_ARCHIVE.name}')

print('Publication disabled by default; no Dataset version is uploaded by Run All.')

# STOP BEFORE DEV/TEST

No rollout or evaluation commands are executed or enabled in this checkpoint-infrastructure notebook. Do not add TEST stages or artifacts to P3 persistence.